In [0]:
def run_data_quality_checks(table, rd_df, dq_rules): ## table = target table. rd_df = retrieved data dataframe. dq_rules = data quality rules
    critical_failures = []
    warning_failures  = []

    table_rules = [r for r in dq_rules if r['table_name'] == table]

    print(f"Data Quality Checks — analytics_lakehouse.{TARGET_SCHEMA}.{table}")

    for rule in table_rules:
        if rule['check_type'] == 'WHERE':
            fail_count = rd_df.where(rule['sql_expression']).count()
        elif rule['check_type'] == 'UNIQUE':
            fail_count = rd_df.groupBy(rule['sql_expression']).count().filter(F.col('count') > 1).count()
        else:
            raise Exception(f"Unsupported check type: {rule['check_type']}")

        if fail_count > 0 and rule['severity'] == 'CRITICAL':
            critical_failures.append({
                'table_name': rule['table_name'],
                'rule_name': rule['rule_name'],
                'fail_count': fail_count
            })
            print(f"CRITICAL {rule['rule_name']}. {fail_count} records failed")
 
        elif fail_count > 0 and rule['severity'] == 'WARNING':
            warning_failures.append({
                'table_name': rule['table_name'],
                'rule_name': rule['rule_name'],
                'fail_count': fail_count
            })
            print(f"WARNING {rule['rule_name']}. {fail_count} records failed")
 
        else:
            print(f"PASSED {rule['rule_name']}")
    
    total = len(table_rules)
    passed = total - len(critical_failures) - len(warning_failures)
 
    print(f"{total} checks | {passed} passed | {len(warning_failures)} warnings | {len(critical_failures)} critical")
 
    # Halt pipeline on CRITICAL failures
    if critical_failures:
        failure_details = ', '.join([f"{f['rule_name']} ({f['fail_count']} rows)" for f in critical_failures])
        raise Exception(f"Data quality validation FAILED for {table}. Critical failures: {failure_details}")
 
    return len(warning_failures)